# Responses API: Scenario 16: Quote-To-Cash (Sequential)

This notebook mirrors `release_room.scenarios.scenario_16_quote_to_cash_sequential` and teaches the **sequential** orchestration pattern with code-defined agents. Scenario id: `scenario-16-quote-to-cash-sequential`.

## 1. Notebook Setup

Run from the project virtual environment after `python -m pip install -e .`. The setup cell also applies the Aptos notebook theme (it falls back to a system sans-serif if Aptos is not installed).

In [ ]:
from pathlib import Path
import sys

def find_project_root(start):
    current = Path(start).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "src" / "release_room").exists():
            return candidate
        nested = candidate / "responses-api-release-room"
        if (nested / "src" / "release_room").exists():
            return nested
    raise RuntimeError("Could not find responses-api-release-room project root.")

PROJECT_ROOT = find_project_root(Path.cwd())
src_path = str(PROJECT_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

PROJECT_ROOT

## 2. Coded Agents, Not Prompt Agents

Every agent in this scenario is a **coded agent**. Two things make that true, and neither relies on a bare instructions prompt:

1. **Code-defined function tools.** Each agent is given real Python callables (from `code_tools.py`) such as `note_observation`, `rate_risk`, `make_checklist`, `extract_action_items`, `tally_votes`, and `compose_summary`. The model invokes this deterministic code to do its work.
2. **Code-defined orchestration.** The agents are wired together by custom `Executor` subclasses inside a `WorkflowBuilder` graph (or a code-driven orchestration builder), so the control flow, routing, aggregation, and state handling are all defined in code.

This mirrors the advanced Microsoft Agent Framework samples, where behaviour comes from executors, typed handlers, and function tools rather than prompt text alone.

## 3. Pattern Concept: Sequential

**Sequential orchestration** runs a fixed pipeline: each stage receives the accumulated work of the stages before it and adds its own contribution. It is the right pattern when every request must pass through the same ordered steps and each step depends on the previous one's output.

## 4. API Fit: Responses vs Invocations

The orchestration is identical across both sample projects; only the API boundary differs.

- **Responses API (this project):** the client sends an OpenAI-style `input`, the scenario is fixed at server startup, and the coded workflow runs server-side behind a stable `/responses` contract.
- **Invocations API (this project):** the client sends a structured job payload and chooses the scenario per request; the handler returns an application-owned JSON result from `/invocations`.

Choose Responses for chat-style clients that expect a stable OpenAI-compatible shape; choose Invocations for systems that route different job types to different scenarios.

## 5. Load The Scenario And Helpers

Each scenario is a single `SCENARIO` object. Helper functions keep the notebook cells short and readable (PEP8-friendly).

In [ ]:
from release_room.scenarios.scenario_16_quote_to_cash_sequential import SCENARIO
from release_room.scenarios import SCENARIOS_BY_ID, get_scenario
from release_room.notebook_helpers import (
    agent_roster,
    apply_notebook_style,
    coded_agent_tool_map,
    default_ollama_kwargs,
    mcp_tool_context,
    pattern_anatomy,
    responses_api_reference,
    scenario_summary,
    workflow_result_to_text,
)

apply_notebook_style()
scenario = SCENARIO
assert SCENARIOS_BY_ID["scenario-16-quote-to-cash-sequential"] is scenario
assert get_scenario("scenario-16-quote-to-cash-sequential") is scenario
scenario_summary(scenario)

## 6. Sample Request

The sample `/responses` payload for this scenario.

In [ ]:
import json
sample_payload = json.loads((PROJECT_ROOT / "samples" / "scenario-16-quote-to-cash-sequential.json").read_text())
sample_payload

## 7. Pattern Anatomy

The framework-level behaviour of this orchestration pattern.

In [ ]:
pattern_anatomy(scenario)

## 8. Flow Diagram

A runtime Mermaid diagram of the orchestration. Dashed links show each coded agent's code tools (and MCP tools where used). The helper renders an image and returns the Mermaid source.

In [ ]:
from release_room.diagram_helpers import display_scenario_flow

from release_room.diagram_helpers import quote_to_cash_flow_diagram
print(quote_to_cash_flow_diagram(scenario).mermaid)
flow = display_scenario_flow(scenario)
flow.mermaid

## 9. MCP Tool Context

These agents are also grounded by the local `quote_to_cash_context` MCP server (FastMCP over stdio; no network, credentials, or writes). Below are the tools it exposes and one deterministic sample call.

In [ ]:
from release_room.mcp_servers import quote_to_cash_context

context = mcp_tool_context(scenario)
print("Server exposes:", quote_to_cash_context.AVAILABLE_TOOLS)
print("Tools used in this scenario:", context["all_tools_used"])
quote_to_cash_context.crm_get_quote_trigger("OPP-5001")

## 10. Agent Roster And Coded Tool Map

The roster lists each agent; the tool map shows the code tools (and any MCP tools) every coded agent can call. No agent is prompt-only.

In [ ]:
{
    "roster": agent_roster(scenario),
    "tool_map": coded_agent_tool_map(scenario),
}

## 11. Advanced Orchestration Internals

Under the hood this scenario is an explicit `WorkflowBuilder` graph: a code-defined `PromptDispatchExecutor` seeds shared state, each role is an `AgentExecutor`, and a `StageGateExecutor` between stages accumulates the running transcript in `WorkflowContext` state before forwarding it to the next stage. A `SequentialOutputExecutor` emits the full transcript.

## 12. Build The Workflow

The same builder path the `/responses` server uses, in-process.

In [ ]:
from release_room.agents import build_ollama_config
from release_room.workflows import build_release_workflow, default_sample_max_tokens

config = build_ollama_config(**default_ollama_kwargs())
max_tokens = default_sample_max_tokens(scenario)  # 500 for Magentic, 250 otherwise
workflow = build_release_workflow(
    "scenario-16-quote-to-cash-sequential",
    model=config.model,
    ollama_host=config.host,
    temperature=config.temperature,
    num_ctx=config.num_ctx,
    max_tokens=max_tokens,
    keep_alive=config.keep_alive,
    think=config.think,
)
workflow

## 13. Live In-Process Run

This calls Ollama. Coded executors and tools run automatically.

In [ ]:
result = await workflow.run(scenario.sample_input)
print(workflow_result_to_text(result)[:5000])

## 14. API Boundary Reference

Compare the in-process workflow with the hosted endpoint.

In [ ]:
responses_api_reference(scenario)

## 15. Experiments

Try changing an agent's `code_tools`, adjusting `max_tokens` or `temperature`, or editing a role brief in the scenario module, then rebuild and compare. You can also swap the orchestration pattern to see how the same coded agents behave under a different graph.